# Future Projection of MSW 

In this file, we attempt to estimate historic quantities of ash generated from the incineration of MSW, and through that estimate their future projections. 


# Historic Data

In this section, the Eurostat data for MSW Generation and Treatment methods is processed, and the historic ash quantities are estimated and saved as csvs.

Data files used:
- env_wasmun_linear.csv: Eurostat MSW GEN and INC data
- country_codes.csv: Saved data for the country codes used in this project
- msw_oecd.csv: OECD MSW GEN and INC data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

/var/folders/bm/tmx1rxns5yvdzj8zkylvh57xfl_lm7/T/ipykernel_12339/473848471.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eu_msw.drop(["DATAFLOW","LAST UPDATE","freq","OBS_FLAG"],axis = 'columns', inplace = True)


## Data Processing

Before estimations can be performed it is necessary to save the data in a processable format and fill in data gaps. In this section, the data is loaded, cleaned, and data gaps are identified. Then OECD data is used to fill in data gaps, whereever possible. Otherwise estimations are used.

### Loading data and selection of relevant metrics

In [ ]:
#Reading MSW data downloaded from Eurostat
msw_data = pd.read_csv("data_downloads/env_wasmun_linear.csv")
country_codes = pd.read_csv("data_saved/country_codes.csv")

In [ ]:
#List of EU27+4 regions, units, waste operations and time periods that are under study
eu27 = country_codes["Code"].values

units = msw_data["unit"].unique()
wst_oper = msw_data["wst_oper"].unique()

time_period = msw_data["TIME_PERIOD"].unique() 

In [ ]:
#Selecting data with E27 regions 

eu_msw = msw_data.loc[(msw_data["geo"].isin(eu27))]
eu_msw.drop(["DATAFLOW","LAST UPDATE","freq","OBS_FLAG"],axis = 'columns', inplace = True)

In [ ]:
#Pivoting table to perform calculations
eu_msw = eu_msw.pivot(index=['geo','TIME_PERIOD','unit'], columns='wst_oper', values='OBS_VALUE')

In [ ]:
#Mapping country code to 3-letter codes
eu_msw = eu_msw.reset_index()
eu_msw["Country"] = eu_msw.merge(country_codes, left_on='geo', right_on = "Code", how = 'left')["NUTS_code"]
eu_msw.drop(['geo'], axis = 'columns', inplace = True)

### Data Cleaning
The Eurostat data contains a lot of NaNs. In this section, we check for data gaps and fill in values instead of NaNs so that the data can be processed.
#### Removing Nans
- If one of the values of DSP_I and RCV_E is NaN, then it is assumed to be zero and DSP_I_RCV_E is a summation of DSP_I, RSV_E.
- If all three are zero, then if the volume for other treatment methods add up to volume of treatment, then everything is zero

In [ ]:
nan_count = eu_msw.isna().sum()

#Cleaning Prepare for Reuse columns
eu_msw.loc[eu_msw['PRP_REU'].isna(),'PRP_REU'] = 0
    
#Cleaning Energy recovery and Incineration data


#Changing NaN values when one of DSP_I, RCV_E is NaN: 
eu_msw.loc[(eu_msw['DSP_I'].isna()) & (eu_msw['RCV_E'].notna()), 'DSP_I']=0
eu_msw.loc[(eu_msw['RCV_E'].isna()) & (eu_msw['DSP_I'].notna()), 'RCV_E']=0
eu_msw['DSP_I_RCV_E']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['DSP_I'].notna()) &
                            (eu_msw['RCV_E'].notna()), eu_msw[['DSP_I','RCV_E']].sum(axis=1),
                            eu_msw['DSP_I_RCV_E']) 


#When all three are NaN, Checking if the volumes of other treatment methods add 
#up, and changing NaN

eu_msw['DSP_I']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['RCY'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['RCY']==eu_msw['TRT']),
                                       0, eu_msw['DSP_I'])
eu_msw['RCV_E']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['RCY'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['RCY']==eu_msw['TRT']),
                                       0, eu_msw['RCV_E'])
eu_msw['DSP_I_RCV_E']=np.where((eu_msw['DSP_I_RCV_E'].isna()) & (eu_msw['RCY'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['RCY']==eu_msw['TRT']),
                                       0, eu_msw['DSP_I_RCV_E'])

#Cleaning Recycling data

#Similar to incineration, changing NaN when one of the two recycling processes is NaN
eu_msw.loc[(eu_msw['RCY_C_D'].isna()) & (eu_msw['RCY_M'].notna()), 'RCY_C_D']=0
eu_msw.loc[(eu_msw['RCY_M'].isna()) & (eu_msw['RCY_C_D'].notna()), 'RCY_M']=0
eu_msw['RCY']=np.where((eu_msw['RCY'].isna()) & (eu_msw['RCY_C_D'].notna()) &
                            (eu_msw['RCY_M'].notna()), eu_msw[['RCY_C_D','RCY_M']].sum(axis=1),
                            eu_msw['RCY']) 


#When all three are NaN, Checking if the volumes of other treatment methods add 
#up, and changing NaN

eu_msw['RCY_M']=np.where((eu_msw['RCY'].isna()) & (eu_msw['DSP_I_RCV_E'].notna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['DSP_I_RCV_E']==eu_msw['TRT']),
                                       0, eu_msw['RCY_M'])
eu_msw['RCY_C_D']=np.where((eu_msw['DSP_I_RCV_E'].notna()) & (eu_msw['RCY'].isna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['DSP_I_RCV_E']==eu_msw['TRT']),
                                       0, eu_msw['RCY_C_D'])
eu_msw['RCY']=np.where((eu_msw['DSP_I_RCV_E'].notna()) & (eu_msw['RCY'].isna())
                                      & (eu_msw['DSP_L_OTH'].notna()) & 
                                      (eu_msw['DSP_L_OTH']+eu_msw['DSP_I_RCV_E']==eu_msw['TRT']),
                                       0, eu_msw['RCY'])


### Missing Data 

#### OECD data for data filling
Countries in the required time period with missing information are identified first. OECD data is checked for the missing data and used to fill in the gaps if values are found here.

In [ ]:
#Check for missing data
country_check = country_codes["NUTS_code"].iloc[:31]
missing_data = []
for country in country_check:
    for year in np.arange(2010, 2022):
        if eu_msw.loc[(eu_msw["Country"]==country) & (eu_msw["TIME_PERIOD"]==year)].empty:
            missing_data.append([country,year])
            
missing_data = pd.DataFrame(missing_data, columns = ["Country", "TIME_PERIOD"])

In [ ]:
#Loading OECD MSW data and selecting relevant countries, treatment methods, variables, and units of measure
oecd_msw = pd.read_csv("data_downloads/msw_oecd.csv")
oecd_units = oecd_msw["Unit of measure"].unique()
oecd_msw = oecd_msw.loc[oecd_msw["Unit of measure"].isin(oecd_units[1:3])]
oecd_msw = oecd_msw.loc[oecd_msw["REF_AREA"].isin(country_codes["NUTS_code"])]
measure = ['Total waste generated',
          'Material recovery (Recycling + Composting)', 'Composting',
          'Recycling', 'Other recovery',
          'Incineration without energy recovery', 'Memo: total incineration',
          'Landfill', 'Disposal operations',
          'Other disposal', 'Recovery operations',
          'Amounts designated for treatment operations',
          'Incineration with energy recovery']
oecd_msw = oecd_msw.loc[oecd_msw["Measure"].isin(measure)]
oecd_vars = ["REF_AREA","Reference area", "Measure","UNIT_MEASURE", "Unit of measure","TIME_PERIOD", "OBS_VALUE","UNIT_MULT"]
oecd_msw = oecd_msw[oecd_vars]

oecd_msw_percap = oecd_msw.loc[oecd_msw["UNIT_MEASURE"]=='KG_PS'] 
oecd_msw_tonnes = oecd_msw.loc[oecd_msw["UNIT_MEASURE"]=='T'] 

oecd_msw_tonnes = oecd_msw_tonnes.pivot(index=["REF_AREA","Reference area","UNIT_MEASURE", "Unit of measure","TIME_PERIOD","UNIT_MULT"], columns=['Measure'], values='OBS_VALUE')
oecd_msw_tonnes = oecd_msw_tonnes.reset_index()

oecd_msw_percap = oecd_msw_percap.pivot(index=["REF_AREA","Reference area","UNIT_MEASURE", "Unit of measure","TIME_PERIOD","UNIT_MULT"], columns=['Measure'], values='OBS_VALUE')
oecd_msw_percap = oecd_msw_percap.reset_index()

In [ ]:
#Getting the missing values for UK from OECD data

oecd_uk = oecd_msw_tonnes.loc[(oecd_msw_tonnes["REF_AREA"]=='GBR')&(oecd_msw_tonnes["TIME_PERIOD"].isin(time_period))]
oecd_uk_cap = oecd_msw_percap.loc[(oecd_msw_percap["REF_AREA"]=='GBR')&(oecd_msw_percap["TIME_PERIOD"].isin(time_period))]

#Columns to drop : 'Recovery operations', 'Other recovery','Disposal operations','Other disposal'
map_trt = {'Total waste generated':'GEN',
          'Material recovery (Recycling + Composting)':'RCY', 
          'Composting' : 'RCY_C_D',
          'Recycling': 'RCY_M',
          'Incineration without energy recovery': 'DSP_I', 
          'Memo: total incineration': 'DSP_I_RCV_E',
          'Landfill':'DSP_L_OTH', 
          'Amounts designated for treatment operations': 'TRT',
          'Incineration with energy recovery': 'RCV_E'}

oecd_uk.drop(['Recovery operations', 'Other recovery','Disposal operations','Other disposal', "Unit of measure", "UNIT_MULT", "Reference area"], axis = 'columns', inplace = True)
oecd_uk = oecd_uk.rename(columns = map_trt)
oecd_uk = oecd_uk.rename(columns = {"REF_AREA":'Country',"UNIT_MEASURE":'unit'})
oecd_uk["unit"]='THS_T'
#Prepare for reuse
oecd_uk["PRP_REU"] = 0

oecd_uk_cap.drop([ "Unit of measure", "UNIT_MULT", "Reference area"], axis = 'columns', inplace = True)
oecd_uk_cap = oecd_uk_cap.rename(columns = map_trt)
oecd_uk_cap = oecd_uk_cap.rename(columns = {"REF_AREA":'Country',"UNIT_MEASURE":'unit'})
oecd_uk_cap["unit"]='KG_HAB'
#Prepare for reuse
oecd_uk_cap["PRP_REU"] = 0

In [ ]:
#Separating per cap data and total volume data

eu_msw_cap = eu_msw.loc[eu_msw["unit"] == units[0]]

#selecting bulk values of msw
eu_msw = eu_msw.loc[eu_msw["unit"] == units[1]]

#### Data Filling by Interpolation
When OECD data is also not available, it is checked if the data for neighbouring years are available, and interpolation is used to fill in the data. Thus, for the rows which still contain NaN as missing data has not been filled yet, interpolation is used (except for countries GRC or HRV (since they have some missing values)).

In [ ]:
missing_data['unit'] = 'THS_T'

eu_msw = pd.concat([eu_msw,missing_data.loc[(missing_data['TIME_PERIOD']<2020)&(missing_data['Country']!='GBR')]])
eu_msw.sort_values(by=['Country', 'TIME_PERIOD'], inplace = True, ignore_index = True)


excluded_countries = ['GRC', 'HRV']
interpolate_df = eu_msw[~eu_msw["Country"].isin(excluded_countries)].copy()
interpolate_df = interpolate_df.interpolate()
eu_msw.update(interpolate_df, overwrite = False)

In [ ]:
#doing the same for per cap data

country_check = country_codes["NUTS_code"].iloc[:31]
missing_data_cap = []
for country in country_check:
    for year in np.arange(2010, 2022):
        if eu_msw_cap.loc[(eu_msw_cap["Country"]==country) & (eu_msw_cap["TIME_PERIOD"]==year)].empty:
            missing_data_cap.append([country,year])
            
missing_data_cap = pd.DataFrame(missing_data_cap, columns = ["Country", "TIME_PERIOD"])


eu_msw_cap = pd.concat([eu_msw_cap,missing_data_cap.loc[(missing_data_cap['TIME_PERIOD']<2020)&(missing_data_cap['Country']!='GBR')]])
eu_msw_cap.sort_values(by=['Country', 'TIME_PERIOD'], inplace = True, ignore_index = True)

excluded_countries = ['GRC', 'HRV']
interpolate_df = eu_msw_cap[~eu_msw_cap["Country"].isin(excluded_countries)].copy()
interpolate_df = interpolate_df.interpolate()
eu_msw_cap.update(interpolate_df, overwrite = False)

#### Combining the data to obtain values for EU27+4

In [ ]:
# adding missing values for UK from OECD data
eu_msw = pd.concat([eu_msw[eu_msw["Country"]!='GBR'],oecd_uk], ignore_index = True)

#countries_to_group = ['EU27_2020', 'GBR', 'ISL', 'NOR', 'CHE']
#summing all countries instead of using EU27_2020

countries_to_group = country_codes["NUTS_code"][:-1].to_list()
df_filtered = eu_msw[eu_msw['Country'].isin(countries_to_group)]

# Create a new row for 'EU 27+4' by summing values for specified countries
df_filtered = df_filtered.groupby(['TIME_PERIOD', 'unit'], as_index=False).sum()

df_filtered['Country'] = 'EU27+4'
df_filtered = df_filtered[~df_filtered['TIME_PERIOD'].isin([2020,2021])]

eu_msw = pd.concat([eu_msw, df_filtered], ignore_index=True)

#Saving cleaned data
#eu_msw.to_csv("EU_MSW_Cleaned_Data.csv")


# adding missing values for UK from OECD data
eu_msw_cap = pd.concat([eu_msw_cap[eu_msw_cap["Country"]!='GBR'],oecd_uk_cap], ignore_index = True)

# Saving cleaned data
#eu_msw_cap.to_csv("EU_MSW_percap_Cleaned.csv",index = False)

## Estimation of historic ash quantities

From BAT document the following weight percentages of bottom ash, fly ash, and boiler ash are obtained.
- Bottom Ash = 150-350 kg/t = 15-35 % wt/wt
- Boiler Ash = 20-40 kg/t   = 2-4% wt/wt
- Fly Ash    = 15-60 kg/t   = 1.5-6 % wt/wt
    
They corresponding to the following waste codes within FutuRaM
- Fly ash                   : SLASH_flyAshesWasteInc
- Bottom ash and boiler ash : SLASH_bottomAshesWasteInc

The ash quantities are estimated by combining these wt/wt percentages with the amount of MSW that is sent for incineration. 


In [ ]:
#getting BA/FA for all historic years required for the data structure
        
eu_msw_his = eu_msw.loc[eu_msw["TIME_PERIOD"].isin(np.arange(2010,2022))]
eu_msw_his= eu_msw_his[['GEN','DSP_I_RCV_E','TIME_PERIOD','unit','Country']]
eu_msw_his = eu_msw_his.rename(columns = {'GEN':"MSW_Gen", "DSP_I_RCV_E":"MSW_WasteInc"})  #this step is not needed

#estimating ash quantities

eu_msw_his["SLASH_bottomAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.28
eu_msw_his["SLASH_flyAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.03
#eu_msw_his["SLASH_boilerAshesWasteInc"] = eu_msw_his['MSW_WasteInc']*0.03
eu_msw_his.drop(['MSW_Gen','MSW_WasteInc'], axis = 'columns', inplace = True)

## Data Output Structure
In this section, the data that is estimated is saved according to the format required for the FutuRaM project deliverables.

In [ ]:
# Creating output data structure
columns = ['Waste Stream', 'Country', 'Year', 'Scenario', 'Substance main parent',
           'Stock/Flow ID', 'Value', 'Unit', 'Data Quality', 'Reference', 'Remark 1',
           'Remark 2', 'Remark 3']


eu_msw_his= pd.melt(eu_msw_his, id_vars=['Country','unit','TIME_PERIOD'], value_vars = ['SLASH_bottomAshesWasteInc',
                                'SLASH_flyAshesWasteInc'],var_name='Stock/Flow ID', value_name='Value')

# Changing all values to tonnes
eu_msw_his["Value"] *=1000
eu_msw_his['unit'] = 't'
eu_msw_his = eu_msw_his.rename(columns = {'unit':"Unit"})

# Adding year, waste stream
eu_msw_his = eu_msw_his.rename(columns = {'TIME_PERIOD':'Year'})
eu_msw_his['Waste Stream'] = 'SLASH'

# Adding LowKey codes
subs_main_parent = {'SLASH_flyAshesWasteInc':'19 01 13','SLASH_bottomAshesWasteInc':'19 01 11',
                    'SLASH_boilerAshesWasteInc':'19 01 15'}
eu_msw_his['Substance main parent'] = eu_msw_his['Stock/Flow ID'].map(subs_main_parent)

# Rearrange columns

eu_msw_his[['Scenario']] = 'OBS'
eu_msw_his[['Reference']] = np.nan
eu_msw_his[['Remark 2']] = np.nan

eu_msw_his.loc[eu_msw_his["Country"]=='GBR',['Remark 1']] = 'Estimated from OECD MSW incineration quantities'
conditions = (((eu_msw_his["Country"]=='DNK')&(eu_msw_his["Year"]==2010)) | ((eu_msw_his["Country"]=='IRL')&(eu_msw_his["Year"]==2013)) | ((eu_msw_his["Country"]=='IRL')&(eu_msw_his["Year"]==2015)) | ((eu_msw_his["Country"]=='ISL')&(eu_msw_his["Year"]==2019)))
eu_msw_his.loc[conditions,['Remark 1']]='Missing data, estimated from Eurostat MSW incineration quantities of neighbouring years '
eu_msw_his.loc[(eu_msw_his["Country"]!='GBR')& ~conditions,['Remark 1']]='Estimated from Eurostat MSW incineration quantities'
eu_msw_his[['Remark 3']] = 'Sowmya Ravisandiran'
eu_msw_his[['Data Quality']] = 2
eu_msw_his = eu_msw_his[columns]
eu_msw_his.drop(eu_msw_his.loc[eu_msw_his["Country"]=='EU27_2020'].index, inplace = True)

# Save the data
#eu_msw_his.to_csv('Data_Structure_Task4.1_Task4.2.csv')